In [1]:
# conda activate genomic_tools

import os
import sys
import pysam
import pickle
import pandas as pd
from Bio.Seq import Seq
from pyfaidx import Fasta
from collections import defaultdict

sys.path.append("code")

from modified_functions import *

pd.set_option('display.max_colwidth', None)

In [2]:
# Sequence-based domain detection (run HMMER/ELM directly on the exon peptide) sidesteps coordinates — a match is a match regardless of numbering. 
# But position-based cross-referencing (UniProt features, mapping full-protein InterProScan hits onto exons, isoform diffs) genuinely needs a correct, shared Met1.

In [3]:
def score_transcript(t, x):
    tag = str(x['transcript_tag'])
    return (
        int("MANE_Select" in tag),
        int("MANE_Plus_Clinical" in tag),
        int("appris_principal" in tag),
        int("basic" in tag),
        int("CCDS" in tag),
        int("GENCODE_Primary" in tag),
        x['coding_nt_length'] if "coding_nt_length" in x.keys() else 0,    # prefer longer (more complete) CDS
        t   # deterministic alphabetical tiebreak
    ) 

def _cds_rows(obj):
    if obj is None:
        return None
    if hasattr(obj, "iterrows"):
        return [{'start': int(r['start']), 'end': int(r['end']), 'frame': int(r['frame'])}
                for _, r in obj.iterrows()]
    return [{'start': int(c['start']), 'end': int(c['end']), 'frame': int(c['frame'])} for c in obj]
 
def _make_protein_sequence(transcript, exon_dict, transcript_chrom, cds_by_transcript, genome, skip=False):
    cds = cds_by_transcript.get(transcript)
    if cds is None:
        return None

    strand = exon_dict.get('strand', '+')
    cds_rows = _cds_rows(cds)
    if strand == '+':
        cds_rows = sorted(cds_rows, key=lambda c: c['start'])
    else:
        cds_rows = sorted(cds_rows, key=lambda c: c['start'], reverse=True)

    coding_seq = ''
    for c in cds_rows:
        if skip and c['start'] == exon_dict.get('exon_cds_start') \
               and c['end']   == exon_dict.get('exon_cds_end'):
            continue
        seq = genome.fetch(transcript_chrom, c['start'] - 1, c['end'])
        if strand == '-':
            seq = str(Seq(seq).reverse_complement())
        coding_seq += seq

    if not coding_seq:
        return None
    protein = str(Seq(coding_seq).translate(to_stop=True))
    return protein if protein else None

def get_coding_nt_length(transcript, cds_by_transcript):
    cds = cds_by_transcript.get(transcript)
    if cds is None:
        return 0
    return sum(c['end'] - c['start'] + 1 for c in _cds_rows(cds))

In [9]:
# Load GTF

exclude = ""
gene_name = "gene_name"
gene_type = "all"
no_trim_id = False
gene_type_tag = "gene_type"
transcript_type_tag = "transcript_type"

gtf_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v46.annotation.gtf"
gtf = process_gtf(gtf_file, exclude, gene_name, no_trim_id, gene_type_tag, transcript_type_tag)

# gtf_cds = gtf[gtf.feature == "CDS"]
# cds_by_transcript = {t: grp for t, grp in gtf_cds.groupby('transcript')}  # for transcript coding sequences lookup

gtf_exon = gtf[gtf.feature == "exon"]
exons_by_transcript = {t: grp for t, grp in gtf_exon.groupby('transcript')}  # for transcript lookup

# with open("data/gencode.v46.annotation_cds_by_transcript.pkl", "wb") as f:
#     pickle.dump(cds_by_transcript, f)

with open("data/gencode.v46.annotation_exons_by_transcript.pkl", "wb") as f:
    pickle.dump(exons_by_transcript, f)

Processing GTF file...


INFO:root:Extracted GTF attributes: ['gene_id', 'gene_type', 'gene_name', 'level', 'tag', 'transcript_id', 'transcript_type', 'transcript_name', 'transcript_support_level', 'havana_transcript', 'exon_number', 'exon_id', 'hgnc_id', 'havana_gene', 'ont', 'protein_id', 'ccdsid', 'artif_dupl']


In [ ]:
event_dicts = pickle.load(open('data/event_dicts.pkl', 'rb'))
event_info = pickle.load(open('data/event_info.pkl', 'rb'))

cds_by_transcript= pickle.load(open("data/gencode.v46.annotation_cds_by_transcript.pkl", "rb"))
exons_by_transcript = pickle.load(open("data/gencode.v46.annotation_exons_by_transcript.pkl", "rb"))
genome_fasta = pysam.FastaFile("/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/GRCh38.primary_assembly.genome.fa") 

# Get all significant splicing events
signif_events = []
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        for idx, _ in signif_exons_df.iterrows():
            if idx not in signif_events:
                signif_events.append(idx)

In [10]:
# recording coding/AA position of the exon for BOTH contexts:
# (1) the inclusion event and 
# (2) sibling event (if they exist)

event_by_coords = {(ed['es'], ed['ee']): ed['event'] for ed in event_dicts} # for boundary siblings: one event per unique (es, ee)

event_by_coords_list = defaultdict(list) # for junction siblings: multiple events can share same (es, ee)
for ed in event_dicts:
    event_by_coords_list[(ed['es'], ed['ee'])].append(ed['event'])

tags_to_exclude = {'mRNA_start_NF', 'cds_start_NF', 'mRNA_end_NF', 'cds_end_NF'}

def has_incomplete_tag(tag):
    return any(t in str(tag) for t in tags_to_exclude)

MIN_AA = 30

interproscan_targets = {}
event_protein_map  = {}

for ev, rec in event_info.items():
    # only record sequences for significant events
    if ev not in signif_events:
        continue

    # for each signif. event: select best compatible protein-coding transcript
    comp = {t: d for t, d in rec['compatible'].items()
            if isinstance(d.get('overlap_type'), str)
            and d['overlap_type'] != 'noncoding_or_utr'
            and d.get('transcript_type') == 'protein_coding'
            and not has_incomplete_tag(d.get('transcript_tag', ''))}
    if not comp:
        continue
    
    best_t = max(comp, key=lambda t: score_transcript(t, comp[t]))
    d = comp[best_t]
    chrom = rec['meta']['chrom']

    event_protein_map[ev] = {
        'meta': rec['meta'],
        'inclusion': {
            'transcript_id': best_t,
            'aa_start': d['aa_start'],
            'aa_end': d['aa_end'],
            'exon_cds_start': d['exon_cds_start'],
            'exon_cds_end': d['exon_cds_end'],
            'frame_preserving': d['frame_preserving'],
            'clean_start': d['clean_start'],
            'clean_end': d['clean_end']
        },
        'real_skip': None,
        'exon_diff_boundary_siblings': [],
        'exon_diff_junction_siblings': []
    }

    # add sequence of (top-scoring) transcript where the event is SKIPPED
    # prefer siblings (emitted events), fall back to any protein-coding transcript
    skip_candidates = {}
    for t, skip_d in rec['exon_skipped'].items():
        if not isinstance(skip_d, dict):
            continue
        exons = exons_by_transcript.get(t)
        if exons is None:
            continue
        ttype = exons.iloc[0].get('transcript_type', '')
        tag = exons.iloc[0].get('tag', '')
        if ttype != 'protein_coding' or has_incomplete_tag(tag):
            continue
        skip_candidates[t] = {
            'transcript_tag': tag,
            'coding_nt_length': get_coding_nt_length(t, cds_by_transcript),
            'is_called_sibling': skip_d.get('is_called_sibling', False)
        }

    if skip_candidates:
        called = {t: v for t, v in skip_candidates.items() if v['is_called_sibling']}
        pool = called if called else skip_candidates
        best_skip_t = max(pool, key=lambda t: score_transcript(t, pool[t]))
        event_protein_map[ev]['real_skip'] = best_skip_t

    # add SYNTHETIC SKIP sequence: same backbone with cassette exon excised
    if d['frame_preserving']:
        # frame-preserving: manually generate protein product
        skip_seq = _make_protein_sequence(
            best_t, {**d, 'strand': rec['meta']['strand']},
            chrom, cds_by_transcript, genome_fasta, skip=True
        )
        if skip_seq and len(skip_seq) >= MIN_AA:
            skip_id = f"{ev}_synthetic_skip"
            interproscan_targets[skip_id] = skip_seq
            event_protein_map[ev]['synthetic_skip'] = skip_id
    else:
        # frame-shifting: assume transcript produces non-viable protein product
        event_protein_map[ev]['truncation_aa'] = d['aa_start']
        
    # add sequence for (top-scoring) sibling transcript with BOUNDARY VARIANTS
    for sib_t, sib in rec['exon_diff_boundary'].items():
        if not sib.get('is_called_sibling'):
            continue
        sib_ev = event_by_coords.get((sib['start'], sib['end']))
        if not sib_ev or sib_ev not in event_info:
            continue
        sib_comp = {t: d for t, d in event_info[sib_ev]['compatible'].items()
                    if isinstance(d.get('overlap_type'), str)
                    and d['overlap_type'] != 'noncoding_or_utr'
                    and d.get('transcript_type') == 'protein_coding'
                    and not has_incomplete_tag(d.get('transcript_tag', ''))}
        if not sib_comp:
            continue
        
        best_sib_t = max(sib_comp, key=lambda t: score_transcript(t, sib_comp[t]))
        sib_d = sib_comp[best_sib_t]
        entry = {
            'transcript_id':  best_sib_t,
            'aa_start': sib_d['aa_start'],
            'aa_end': sib_d['aa_end'],
            'exon_cds_start': sib_d['exon_cds_start'],
            'exon_cds_end': sib_d['exon_cds_end'],
            'frame_preserving': sib_d['frame_preserving'],
            'clean_start': sib_d['clean_start'],
            'clean_end': sib_d['clean_end'],
        }
        existing = [s['transcript_id'] for s in event_protein_map[ev]['exon_diff_boundary_siblings']]
        if best_sib_t not in existing:
            event_protein_map[ev]['exon_diff_boundary_siblings'].append(entry)
        
    # add sequence for (top-scoring) sibling transcript with JXN VARIANT
    for sib_t, sib in rec['exon_diff_junction'].items():
        if not sib.get('is_called_sibling'):
            continue
        # need to remove parent inclusion transcript since exon boundaries will be identical
        sib_evs = [e for e in event_by_coords_list.get((rec['meta']['es'], rec['meta']['ee']), [])
                   if e != ev and e in event_info]
        for sib_ev in sib_evs:
            if sib_ev not in event_info:
                continue
            sib_comp = {t: d for t, d in event_info[sib_ev]['compatible'].items()
                        if isinstance(d.get('overlap_type'), str)
                        and d['overlap_type'] != 'noncoding_or_utr'
                        and d.get('transcript_type') == 'protein_coding'
                        and not has_incomplete_tag(d.get('transcript_tag', ''))}
            if not sib_comp:
                continue
            best_sib_t = max(sib_comp, key=lambda t: score_transcript(t, sib_comp[t]))
            sib_d = sib_comp[best_sib_t]
            entry = {
                'transcript_id':  best_sib_t,
                'aa_start': sib_d['aa_start'],
                'aa_end': sib_d['aa_end'],
                'exon_cds_start': sib_d['exon_cds_start'],
                'exon_cds_end': sib_d['exon_cds_end'],
                'frame_preserving': sib_d['frame_preserving'],
                'clean_start': sib_d['clean_start'],
                'clean_end': sib_d['clean_end'],
            }
            existing = [s['transcript_id'] for s in event_protein_map[ev]['exon_diff_junction_siblings']]
            if best_sib_t not in existing:
                event_protein_map[ev]['exon_diff_junction_siblings'].append(entry)

/mnt/lareaulab/reliscu/anaconda3/envs/genomic_tools/lib/python3.14/site-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [11]:
# get list of all transcripts to input to InterProScan
transcripts = set()
for info in event_protein_map.values():
    if info.get('inclusion'):
        transcripts.add(info['inclusion']['transcript_id'])
    if info.get('real_skip'):
        transcripts.add(info['real_skip'])
    for sib in info.get('exon_diff_boundary_siblings', []):
        transcripts.add(sib['transcript_id'])
    for sib in info.get('exon_diff_junction_siblings', []):
        transcripts.add(sib['transcript_id'])

In [12]:
len(transcripts)

14948

In [13]:
proteins = Fasta("/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/gencode.v46.pc_translations.fa")

# Build a dict keyed by ENST
protein_by_transcript = {}
for key in proteins.keys():
    parts = key.split("|")
    enst = parts[1].split(".")[0]
    protein_by_transcript[enst] = str(proteins[key])

In [14]:
# write protein sequences

modified_transcript_products = {}
transcript_log = []

with open("data/proteins.fa", "w") as f:
    
    # for real transcripts:
    for key in proteins.keys():
        enst = key.split("|")[1].split(".")[0]
        if enst in transcripts:
            # record the transcripts that have protein sequences
            transcript_log.append(enst)
            # GENCODE protein sequences: index by transcript
            seq = str(proteins[key]).rstrip("*")
            # some transcripts have AA placeholders
            if "X" in seq:
                # track where the placeholder was
                modified_transcript_products[enst] = [i for i, c in enumerate(seq) if c == "X"]
                seq = seq.replace("X", "")
            f.write(f">{enst}\n{seq}\n")
            
    # for synthetic transcripts, previously generated DIY translation:
    for id, seq in interproscan_targets.items():
        f.write(f">{id}\n{seq}\n")

In [ ]:
# modified_transcript_products = {}
# transcript_log = []

# with open("data/proteins.fa", "w") as f:
    
#     for ev, rec in event_protein_map.items():
#         if rec.get('inclusion'):
#             t = rec['inclusion']['transcript_id']
#             if t not in protein_by_transcript.keys():
#                 continue
#             transcripts.add(t)
#             event_protein_map[ev]['inclusion'] =
            
#             if rec.get('real_skip'):
#                 transcripts.add(rec['real_skip'])
#             for sib in rec.get('exon_diff_boundary_siblings', []):
#                 transcripts.add(sib['transcript_id'])
#             for sib in rec.get('exon_diff_junction_siblings', []):
#                 transcripts.add(sib['transcript_id'])

In [32]:
len(transcript_log)

14948

In [33]:
with open("data/event_protein_map.pkl", "wb") as file:
    pickle.dump(event_protein_map, file)
    
with open("data/modified_transcript_products.pkl", "wb") as file:
    pickle.dump(modified_transcript_products, file)